In [37]:
from datetime import datetime
import requests
from datetime import datetime, timezone
import logging
import pandas as pd

from config.app_config import settings
BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

log = logging.getLogger(__name__)



def transform_weather_data(raw_data):
    try:
        if not raw_data:
            return pd.DataFrame()

        df = pd.DataFrame(raw_data)

        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

        df["year"] = df["timestamp"].dt.year
        print(df["year"])
        df["month"] = df["timestamp"].dt.month_name()
        print(df["month"])

        df["day"] = df["timestamp"].dt.day
        print(df["day"])

        df["time"] = df["timestamp"].dt.time
        print(df["time"])

        df["weekday"] = df["timestamp"].dt.day_name()
        print(df["weekday"])

        numeric_cols = [
            "temperature",
            "feels_like",
            "humidity",
            "pressure",
            "wind_speed"
        ]

        for col in numeric_cols:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        df = df[
            [
                "city",
                "country",
                "timestamp",
                "time",
                "day",
                "weekday",
                "month",
                "year",
                "temperature",
                "feels_like",
                "humidity",
                "pressure",
                "wind_speed",
                "weather",
                "description",
            ]
        ]

        return df

    except Exception as e:
        print(f"Transformation failed: {e}")
        return pd.DataFrame()
def test_weather(city):
    print(f"BASE_URL: {BASE_URL}")
    params = {
        "q": city,
        "appid": settings.OPENWEATHER_API_KEY,
        "units": "metric",
    }
    try:
        response = requests.get(BASE_URL, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()

        return {
            "city":        data["name"],
            "country":     data["sys"]["country"],
            "timestamp":   datetime.now(timezone.utc).isoformat(),
            "temperature": data["main"]["temp"],
            "feels_like":  data["main"]["feels_like"],
            "humidity":    data["main"]["humidity"],
            "pressure":    data["main"]["pressure"],
            "wind_speed":  data["wind"]["speed"],
            "weather":     data["weather"][0]["main"],
            "description": data["weather"][0]["description"],
        }

    except requests.exceptions.HTTPError as e:
        log.error("HTTP error fetching weather for %s: %s", city, e)
    except requests.exceptions.ConnectionError:
        log.error("Network error — could not reach OpenWeather API for %s", city)
    except requests.exceptions.Timeout:
        log.error("Request timed out for %s", city)
    except (KeyError, ValueError) as e:
        log.error("Unexpected API response format for %s: %s", city, e)

    return None

def main():
    for city in settings.CITIES:
        print(city)
        results = []
        result = test_weather(city)
        results.append(result)
        print(results)
        return results

raw_data = main()
df =transform_weather_data(raw_data)
df.head()

Nairobi
BASE_URL: https://api.openweathermap.org/data/2.5/weather
[{'city': 'Nairobi', 'country': 'KE', 'timestamp': '2026-05-17T12:37:56.251869+00:00', 'temperature': 21.24, 'feels_like': 21.14, 'humidity': 66, 'pressure': 1013, 'wind_speed': 3.45, 'weather': 'Rain', 'description': 'light rain'}]
0    2026
Name: year, dtype: int32
0    May
Name: month, dtype: object
0    17
Name: day, dtype: int32
0    12:37:56.251869
Name: time, dtype: object
0    Sunday
Name: weekday, dtype: object


,city,country,timestamp,time,day,weekday,month,year,temperature,feels_like,humidity,pressure,wind_speed,weather,description
0,Nairobi,KE,2026-05-17 12:37:56.251869+00:00,12:37:56.251869,17,Sunday,May,2026,21.24,21.14,66,1013,3.45,Rain,light rain


In [36]:
df.head()

,city,country,timestamp,time,day,weekday,month,year,temperature,feels_like,humidity,pressure,wind_speed,weather,description
0,Nairobi,KE,2026-05-17 12:35:36.686279+00:00,12:35:36.686279+00:00,17,Sunday,May,2026,21.24,21.14,66,1013,3.45,Rain,light rain
